# Experimentos 1 e 2 v2 — Versao Final

**Modelos locais (Optuna completo):**
- Llama 3.1 8B — ja completo
- Mistral 7B — ja completo
- Qwen 2.5 7B — ja completo
- Gemma 2 9B — ja completo

**Modelos API (params fixos — limitacao do plano gratuito documentada):**
- Gemini 2.0 Flash

**Nota:** APIs gratuitas (Groq e Gemini) impõem limites de tokens por minuto
incompativeis com o Optuna. O Optuna foi aplicado nos modelos locais.
Para o Gemini, foram utilizados hiperparametros fixos (temp=0.1, max_chars=1500),
conforme orientacao dos coordenadores.

**Secrets necessarios:** HF_TOKEN, googleAI_key

**Upload necessario:** ck1_llama.csv, ck1_mistral.csv, ck1_qwen.csv,
ck2_llama.csv, ck2_mistral.csv, ck2_qwen.csv, ck1_gemma.csv, ck2_gemma.csv

Ambiente de execucao -> Alterar tipo -> GPU A100

In [ ]:
# Celula 1 - Instalacao
!pip install -q transformers accelerate bitsandbytes gdown scikit-learn matplotlib peft optuna google-genai
print('Instalado!')

In [ ]:
# Celula 2 - Imports e autenticacao
import re, gc, json, time, random, os
import numpy as np
import pandas as pd
import torch
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    cohen_kappa_score, f1_score
)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata
from google import genai
from google.genai import types
from huggingface_hub import login

optuna.logging.set_verbosity(optuna.logging.WARNING)

HF_TOKEN   = userdata.get('HF_TOKEN')
GOOGLE_KEY = userdata.get('googleAI_key')

login(token=HF_TOKEN)
gemini_client = genai.Client(api_key=GOOGLE_KEY)

GEMINI_SLEEP = 6.0


def chamada_com_retry(fn, max_tentativas=6):
    for tentativa in range(max_tentativas):
        try:
            return fn()
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'rate' in msg.lower() or 'quota' in msg.lower():
                espera = (2 ** tentativa) + random.uniform(0, 2)
                print(f'  Rate limit — aguardando {espera:.1f}s')
                time.sleep(espera)
            else:
                raise
    raise RuntimeError('Falhou apos tentativas de retry')


print('Autenticado!')

In [ ]:
# Celula 3 - Dataset e K-Fold
import gdown

gdown.download(
    'https://drive.google.com/uc?id=1zg9n7EUDWKfu6t_f3aYDS_QroYXGNGwd',
    'meu_dataset.csv', quiet=False
)
df_enem = pd.read_csv('meu_dataset.csv')


def limpar(texto):
    if pd.isna(texto): return ''
    texto = str(texto).strip("[]'\" ")
    texto = texto.replace('\n', ' ')
    texto = re.sub(r'\[[A-Z/]+\]', '', texto)
    texto = re.sub(r'\{[a-z]+\}', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()


df_enem['essay_limpo'] = df_enem['essay'].apply(limpar)
df_enem = df_enem[df_enem['score'] > 0].reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
df_enem['fold'] = -1
for i, (tr, te) in enumerate(skf.split(df_enem, df_enem['score'])):
    df_enem.loc[te, 'fold'] = i

df_teste = df_enem[df_enem['fold'] == 0].reset_index(drop=True)
print(f'Total: {len(df_enem)} | Teste (fold 0): {len(df_teste)}')

In [ ]:
# Celula 4 - Exemplos few-shot e prompts

df_tr = df_enem[df_enem['fold'] != 0]
EX_BAIXO = df_tr[df_tr['score'].between(80, 300)].iloc[0]
EX_MEDIO = df_tr[df_tr['score'].between(400, 600)].iloc[0]
EX_ALTO  = df_tr[df_tr['score'].between(700, 1000)].iloc[0]
EXEMPLOS = [EX_BAIXO, EX_MEDIO, EX_ALTO]

print('Exemplos few-shot selecionados do treino:')
for ex in EXEMPLOS:
    print(f'  score={ex["score"]} | C1={ex["c1"]} C2={ex["c2"]} C3={ex["c3"]} C4={ex["c4"]} C5={ex["c5"]}')


def gerar_prompt_zs(redacao, max_chars=None):
    if max_chars:
        redacao = redacao[:max_chars]
    return (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie a redacao seguindo a escala oficial: 0, 40, 80, 120, 160 ou 200 pontos por competencia.\n\n'
        'COMPETENCIAS:\n'
        '- C1 (Norma Culta): dominio da norma padrao (0-200)\n'
        '- C2 (Tema/Estrutura): adequacao ao tema e estrutura (0-200)\n'
        '- C3 (Argumentacao): selecao e organizacao de argumentos (0-200)\n'
        '- C4 (Coesao): uso de mecanismos linguisticos (0-200)\n'
        '- C5 (Proposta de Intervencao): proposta com agente, acao, meio, efeito (0-200)\n\n'
        'REDACAO:\n' + redacao + '\n\n'
        'Responda APENAS com o JSON:\n'
        '{"C1": valor, "C2": valor, "C3": valor, "C4": valor, "C5": valor, "Nota_Total": soma}'
    )


def gerar_prompt_fs(redacao, max_chars=None, n_exemplos=3):
    if max_chars:
        redacao = redacao[:max_chars]
    bloco = ''
    for j, ex in enumerate(EXEMPLOS[:n_exemplos]):
        notas_ex = (
            '{"C1": ' + str(ex['c1']) +
            ', "C2": ' + str(ex['c2']) +
            ', "C3": ' + str(ex['c3']) +
            ', "C4": ' + str(ex['c4']) +
            ', "C5": ' + str(ex['c5']) +
            ', "Nota_Total": ' + str(ex['score']) + '}'
        )
        bloco += (
            '--- EXEMPLO ' + str(j + 1) +
            ' (score=' + str(ex['score']) + ') ---\n'
            'REDACAO: "' + ex['essay_limpo'][:300] + '..."\n'
            'AVALIACAO: ' + notas_ex + '\n\n'
        )
    return (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie usando a escala: 0, 40, 80, 120, 160 ou 200 por competencia.\n\n'
        'COMPETENCIAS:\n'
        '- C1 (Norma Culta): 0-200\n'
        '- C2 (Tema/Estrutura): 0-200\n'
        '- C3 (Argumentacao): 0-200\n'
        '- C4 (Coesao): 0-200\n'
        '- C5 (Proposta de Intervencao): 0-200\n\n'
        'EXEMPLOS AVALIADOS POR HUMANOS:\n' + bloco +
        'REDACAO A AVALIAR:\n"' + redacao + '"\n\n'
        'Responda APENAS com o JSON:\n'
        '{"C1": valor, "C2": valor, "C3": valor, "C4": valor, "C5": valor, "Nota_Total": soma}'
    )


print('Prompts carregados!')

In [ ]:
# Celula 5 - Funcoes auxiliares

def extrair_notas(resposta):
    try:
        texto = re.sub(r'```json|```', '', str(resposta))
        i = texto.find('{')
        j = texto.rfind('}') + 1
        if i == -1 or j <= i: return None
        dados = json.loads(texto[i:j])
        comps = ['C1', 'C2', 'C3', 'C4', 'C5']
        if all(c in dados for c in comps):
            if max(dados[c] for c in comps) <= 20:
                for c in comps: dados[c] *= 10
            dados['Nota_Total'] = sum(dados[c] for c in comps)
            return dados
        if 'Nota_Total' in dados: return dados
    except: pass
    return None


def extrair_notas_markdown(resposta):
    try:
        texto = str(resposta)
        comps = {}
        for c in ['C1', 'C2', 'C3', 'C4', 'C5']:
            m = re.search(rf'{c}[^:]*:\s*(\d+)', texto)
            if m: comps[c] = int(m.group(1))
        if len(comps) == 5:
            if max(comps.values()) <= 20:
                for c in comps: comps[c] *= 10
            comps['Nota_Total'] = sum(comps.values())
            return comps
        m = re.search(r'(?:Total|Nota\s*Total|Nota\s*Final)[^\d]*(\d{3,4})', texto, re.I)
        if m: return {'Nota_Total': int(m.group(1))}
    except: pass
    return None


def extrair(resposta):
    return extrair_notas(resposta) or extrair_notas_markdown(resposta)


def calcular_metricas(y_true, y_pred, nome):
    yt = np.array(y_true, dtype=float)
    yp = np.array(y_pred, dtype=float)
    mae  = mean_absolute_error(yt, yp)
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    qwk = cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: qwk = float('nan')
    try:    f1  = f1_score(disc(yt), disc(yp), average='weighted', zero_division=0)
    except: f1  = float('nan')
    print(f'\n{"="*55}')
    print(f'METRICAS — {nome}')
    print(f'{"="*55}')
    print(f'  Amostras : {len(yt)}')
    print(f'  MAE      : {mae:.4f}')
    print(f'  RMSE     : {rmse:.4f}')
    print(f'  QWK      : {qwk:.4f}')
    print(f'  F1 Score : {f1:.4f}')
    print(f'{"="*55}')
    return {'modelo': nome, 'mae': mae, 'rmse': rmse, 'qwk': qwk, 'f1': f1, 'n': len(yt)}


print('Funcoes auxiliares carregadas!')

In [ ]:
# Celula 6 - Inferencia local e Gemini

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)


def carregar_modelo(nome_modelo):
    print(f'Carregando {nome_modelo}...')
    tok = AutoTokenizer.from_pretrained(nome_modelo, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(
        nome_modelo, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True
    )
    m.eval()
    print('Carregado!')
    return tok, m


def liberar(m, tok):
    del m, tok
    gc.collect()
    torch.cuda.empty_cache()
    print('Memoria GPU liberada.')


def inf_local(tok, m, prompt, temp=0.1, max_tok=300):
    msgs = [{'role': 'user', 'content': prompt}]
    try:
        txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except:
        txt = prompt
    inp = tok(txt, return_tensors='pt', truncation=True, max_length=3072).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inp, max_new_tokens=max_tok,
            temperature=temp, do_sample=True,
            pad_token_id=tok.pad_token_id
        )
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def inf_gemini(prompt, temp=0.1, max_tok=300):
    def fn():
        resp = gemini_client.models.generate_content(
            model='gemini-2.0-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=temp,
                max_output_tokens=max_tok,
                response_mime_type='application/json'
            )
        )
        return resp.text
    result = chamada_com_retry(fn)
    time.sleep(GEMINI_SLEEP)
    return result


print('Funcoes de inferencia carregadas!')

In [ ]:
# Celula 7 - Optuna e runner com checkpoint

def obj_optuna(trial, fn_inf, tipo='ZS', n_val=10):
    temp      = trial.suggest_float('temp', 0.01, 0.5)
    max_chars = trial.suggest_int('max_chars', 500, 2000, step=250)
    if tipo == 'FS':
        n_exemplos = trial.suggest_int('n_exemplos', 1, 3)

    df_val = df_enem[df_enem['fold'] == 1].head(n_val)
    yp, yt = [], []
    for _, row in df_val.iterrows():
        try:
            if tipo == 'ZS':
                prompt = gerar_prompt_zs(row['essay_limpo'], max_chars=max_chars)
            else:
                prompt = gerar_prompt_fs(row['essay_limpo'], max_chars=max_chars, n_exemplos=n_exemplos)
            resp  = fn_inf(prompt, temp)
            notas = extrair(resp)
            if notas and 'Nota_Total' in notas:
                yp.append(notas['Nota_Total'])
                yt.append(row['score'])
        except:
            pass
    if len(yp) < 5:
        raise optuna.TrialPruned()
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    return cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: return -1.0


def rodar_optuna(fn_inf, nome, tipo='ZS', n_trials=10):
    print(f'Optuna: otimizando {nome} ({tipo}) — {n_trials} trials...')
    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)
    )
    study.optimize(
        lambda t: obj_optuna(t, fn_inf, tipo=tipo),
        n_trials=n_trials,
        catch=(Exception,)
    )
    print(f'Melhores params: {study.best_params}')
    return study.best_params


def rodar_experimento(nome, fn_inf, params, ckpt_csv, tipo='ZS'):
    temp       = params.get('temp', 0.1)
    max_chars  = params.get('max_chars', None)
    n_exemplos = params.get('n_exemplos', 3)

    if os.path.exists(ckpt_csv):
        df_ck     = pd.read_csv(ckpt_csv)
        ja_feitos = set(df_ck['index_redacao'].tolist())
        print(f'Checkpoint: {len(ja_feitos)} ja processadas para {nome}')
    else:
        df_ck     = pd.DataFrame(columns=['index_redacao', 'score', 'pred_total'])
        ja_feitos = set()

    pendentes = df_teste[~df_teste.index.isin(ja_feitos)]
    print(f'Pendentes: {len(pendentes)}/{len(df_teste)}')

    novos = []
    for idx, (i, row) in enumerate(pendentes.iterrows()):
        try:
            if tipo == 'ZS':
                prompt = gerar_prompt_zs(row['essay_limpo'], max_chars=max_chars)
            else:
                prompt = gerar_prompt_fs(row['essay_limpo'], max_chars=max_chars, n_exemplos=n_exemplos)
            resp  = fn_inf(prompt, temp)
            notas = extrair(resp)
            pred  = notas['Nota_Total'] if notas else None
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': pred})
        except:
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': None})

        if (idx + 1) % 50 == 0:
            df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
            df_ck.to_csv(ckpt_csv, index=False)
            novos = []
            print(f'  Checkpoint: {idx + 1 + len(ja_feitos)}/{len(df_teste)}')

    if novos:
        df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
        df_ck.to_csv(ckpt_csv, index=False)

    df_v = df_ck.dropna(subset=['pred_total'])
    print(f'Validas: {len(df_v)}/{len(df_teste)}')
    if len(df_v) > 0:
        return calcular_metricas(df_v['score'].tolist(), df_v['pred_total'].tolist(), nome)
    return None


print('Optuna e runner carregados!')

## Recuperar resultados dos modelos ja completos

In [ ]:
# Celula 8 - Recuperar metricas dos modelos locais ja completos

resultados_anteriores = {}

for nome_modelo, ck_zs, ck_fs in [
    ('Llama 3.1 8B', 'ck1_llama.csv',   'ck2_llama.csv'),
    ('Mistral 7B',   'ck1_mistral.csv', 'ck2_mistral.csv'),
    ('Qwen 2.5 7B',  'ck1_qwen.csv',    'ck2_qwen.csv'),
    ('Gemma 2 9B',   'ck1_gemma.csv',   'ck2_gemma.csv'),
]:
    for tipo, ckpt in [('Zero-Shot', ck_zs), ('Few-Shot', ck_fs)]:
        if os.path.exists(ckpt):
            df_ck = pd.read_csv(ckpt)
            df_v  = df_ck.dropna(subset=['pred_total'])
            if len(df_v) > 0:
                chave = f'{nome_modelo} ({tipo})'
                resultados_anteriores[chave] = calcular_metricas(
                    df_v['score'].tolist(),
                    df_v['pred_total'].tolist(),
                    chave
                )
        else:
            print(f'ATENCAO: {ckpt} nao encontrado.')

print(f'\nTotal de resultados recuperados: {len(resultados_anteriores)}/8')

## Gemini 2.0 Flash (API)

Params fixos conforme orientacao dos coordenadores — plano gratuito nao suporta Optuna.

In [ ]:
# Celula 9 - Gemini 2.0 Flash (params fixos — sem Optuna)
# Justificativa: plano gratuito da API Google impoem rate limit incompativel
# com o Optuna (multiplas chamadas por trial). Documentado como limitacao tecnica.

fn_gem = lambda p, t: inf_gemini(p, temp=t)

PARAMS_GEM = {'temp': 0.1, 'max_chars': 1500}
PARAMS_GEM_FS = {'temp': 0.1, 'max_chars': 1500, 'n_exemplos': 3}

# Zero-Shot
res_gem_zs = rodar_experimento(
    'Gemini 2.0 Flash (Zero-Shot)', fn_gem,
    PARAMS_GEM, 'ck1_gem.csv', tipo='ZS'
)

# Few-Shot
res_gem_fs = rodar_experimento(
    'Gemini 2.0 Flash (Few-Shot)', fn_gem,
    PARAMS_GEM_FS, 'ck2_gem.csv', tipo='FS'
)

## Resultados Consolidados — Experimentos 1 e 2

In [ ]:
# Celula 10 - Consolidacao final
import matplotlib.pyplot as plt

novos = [res_gem_zs, res_gem_fs]
todos = list(resultados_anteriores.values()) + [r for r in novos if r is not None]
df_res = pd.DataFrame(todos).sort_values('qwk', ascending=False).reset_index(drop=True)

print('\n' + '='*75)
print(f'{"Modelo":<42} {"N":>5} {"MAE":>7} {"RMSE":>7} {"QWK":>7} {"F1":>7}')
print('-'*75)
for _, row in df_res.iterrows():
    print(f'{row["modelo"]:<42} {int(row["n"]):>5} {row["mae"]:>7.3f} {row["rmse"]:>7.3f} {row["qwk"]:>7.3f} {row["f1"]:>7.3f}')
print('='*75)

df_res.to_csv('resultados_exp1_exp2_final.csv', index=False)
print('\nCSV salvo: resultados_exp1_exp2_final.csv')

In [ ]:
# Celula 11 - Graficos comparativos

cores = ['#4C72B0' if 'Zero-Shot' in m else '#DD8452' for m in df_res['modelo']]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle(
    'Experimentos 1 e 2 v2 — Zero-Shot vs Few-Shot com Optuna\nResultados por Modelo e Metrica',
    fontsize=14, fontweight='bold'
)

for ax, (titulo, coluna) in zip(axes.flatten(),
        [('MAE (menor = melhor)', 'mae'),
         ('RMSE (menor = melhor)', 'rmse'),
         ('QWK (maior = melhor)', 'qwk'),
         ('F1 Score (maior = melhor)', 'f1')]):
    barras = ax.barh(df_res['modelo'], df_res[coluna], color=cores, edgecolor='white')
    for b in barras:
        w = b.get_width()
        ax.text(w + 0.002, b.get_y() + b.get_height() / 2,
                f'{w:.3f}', va='center', ha='left', fontsize=8)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.grid(axis='x', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

from matplotlib.patches import Patch
legenda = [Patch(color='#4C72B0', label='Zero-Shot'), Patch(color='#DD8452', label='Few-Shot')]
fig.legend(handles=legenda, loc='lower center', ncol=2, fontsize=11,
           frameon=False, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout()
plt.savefig('grafico_exp1_exp2_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvo: grafico_exp1_exp2_final.png')